# Chapter 10 — Distribution functions of actions and self-consistent galaxy models

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/galactic-dynamics-tutorial/blob/main/notebooks/10_action_dfs.ipynb)

**Goal.** Build complete equilibrium models of galaxies: every component (dark halo, disk) is described by a
distribution function of the actions, and the potential is generated by the total density of all components.
This is how AGAMA models are made, and how the Milky Way is modelled from Gaia data.

**You will learn**
1. why DFs are written as functions of the actions, $f(\mathbf J)$ with $\mathbf J = (J_r, J_z, J_\phi)$ the radial, vertical and azimuthal
   actions in kpc km/s (Chapter 9), and two standard families: the **double power law**
   (spheroids, halos) and the **quasi-isothermal** DF (disks);
2. the **self-consistent iteration**: DF → density → potential → actions → density …, and how to check convergence;
3. to build a **disk + halo** galaxy and read off its rotation curve, vertical structure and kinematics;
4. that a halo with a fixed $f(\mathbf J)$ **contracts** when a disk grows inside it.

**Units.** $M_\odot$ (solar mass), kpc, km/s; actions in kpc km/s.

In [ ]:
import sys, importlib.util
if importlib.util.find_spec('agama') is None:      # install AGAMA on Colab (see Chapter 0)
    v = f'cp{sys.version_info.major}{sys.version_info.minor}'
    !pip install -q "https://github.com/vasilybelokurov/galactic-dynamics-tutorial/releases/download/agama-wheels/agama-1.0-{v}-{v}-linux_x86_64.whl"
import agama

agama.setUnits(mass=1, length=1, velocity=1)       # Msun, kpc, km/s -- before creating any AGAMA object
G = agama.G

import numpy as np
import matplotlib.pyplot as plt

def vcirc(pot, R):
    """Midplane circular speed sqrt(-R F_R) (Chapter 0 §0.4)."""
    R = np.atleast_1d(R)
    return np.sqrt(-R * pot.force(np.column_stack([R, 0 * R, 0 * R]))[:, 0])

def on_x_axis(r):
    """Points (r, 0, 0) as an (N, 3) array, for evaluating densities and potentials along a line."""
    r = np.atleast_1d(r)
    return np.column_stack([r, 0 * r, 0 * r])

## 10.1 Why distribution functions of actions?

By the Jeans theorem (Chapter 6), any DF that depends only on integrals of motion describes a steady state.
Chapter 8 used $f(E)$, a function of the energy $E$, and $f(Q)$, a function of the Osipkov–Merritt variable $Q$ (relative energy minus
$L^2/2r_a^2$, with $L$ the angular momentum and $r_a$ the anisotropy radius). Actions are better integrals for building galaxies:

- they label orbits in any (axisymmetric) potential, so one formula works for halos, disks and bulges;
- they are **adiabatic invariants** (Chapter 9): if the potential changes slowly, e.g. because another component
  is added, $f(\mathbf J)$ stays the same while $f(E)$ would not. A DF of actions can be fixed once and the
  potential adjusted around it;
- their physical meaning is simple: $J_r$ and $J_z$ measure radial and vertical excursions, $J_\phi = L_z$ (the angular momentum about the $z$ axis) the rotation.

**Double power law** (for spheroidal components, Posti et al. 2015; Vasiliev 2019):

$$ f(\mathbf J) = \frac{M}{(2\pi J_0)^3}\left[1 + \left(\frac{J_0}{h(\mathbf J)}\right)^{\eta}\right]^{\Gamma/\eta}
   \left[1 + \left(\frac{g(\mathbf J)}{J_0}\right)^{\eta}\right]^{-B/\eta} \times (\text{optional cutoff}), $$

where $M$ is the total mass, $J_0$ the break action (`J0`), $\eta$ the sharpness of the transition between the two power laws
(`steepness`), and $h$ and $g$ are linear combinations of the actions, $h = h_r J_r + h_z J_z + (3 - h_r - h_z)|J_\phi|$ and
$g = g_r J_r + g_z J_z + (3 - g_r - g_z)|J_\phi|$, whose coefficients (`coefJrIn`, `coefJzIn`, `coefJrOut`, `coefJzOut`) set the
anisotropy and flattening; equal coefficients of 1 give an isotropic sphere. Below the break action, $f \propto J^{-\Gamma}$ (`slopeIn`);
above it, $f \propto J^{-B}$ (`slopeOut`), where $J$ stands for the size of the actions. For a self-consistent model, a density slope
$\rho \propto r^{-\gamma}$ in the centre corresponds to $\Gamma = (6 - \gamma)/(4 - \gamma)$, and $\rho \propto r^{-\beta}$ outside to $B = 2\beta - 3$
(AGAMA reference, §2.5.1); here $\gamma$ and $\beta$ are density slopes (not the anisotropy $\beta$ of Chapter 7).
We want an NFW-like halo ($\gamma = 1$, $\beta \approx 3$), so $\Gamma = 5/3$ and $B$ a little above 3.

## 10.2 A self-consistent halo

A DF alone does not define a model: to know the density we need the actions, and to know the actions we need the
potential, which depends on the density. The **self-consistent iteration** breaks the circle:

1. guess a density (here a Hernquist profile) and compute its potential;
2. compute the actions in this potential, and the density of the DF, $\rho(\mathbf x) = \int f(\mathbf J(\mathbf x, \mathbf v))\, d^3v$;
3. compute the potential of this new density (Poisson's equation, via a multipole expansion);
4. repeat from 2 until nothing changes.

AGAMA's `SelfConsistentModel` does steps 2–3 in each call to `iterate()`. A `Component` holds a DF (or a fixed
density) and the grid on which its density is computed. We follow the density at several radii.

In [ ]:
df_halo = agama.DistributionFunction(type='DoublePowerLaw', mass=1e12, J0=5000., slopeIn=1.6, slopeOut=3.2,
                                     Jcutoff=1e5)    # Jcutoff truncates the halo at large actions
print(f'mass of the DF: {df_halo.totalMass():.3e} Msun')   # known before any iteration: an integral over actions

def halo_model(df):
    """A SelfConsistentModel with one spherical DF component, started from a Hernquist guess."""
    model = agama.SelfConsistentModel(rminSph=0.05, rmaxSph=1000, sizeRadialSph=40, lmaxAngularSph=0,  # potential grid
                                      verbose=False)                 # no progress messages
    guess = agama.Density(type='Dehnen', gamma=1, mass=df.totalMass(), scaleRadius=30.)
    model.components.append(agama.Component(df=df, density=guess, disklike=False,
                                            rminSph=0.05, rmaxSph=500, sizeRadialSph=40, lmaxAngularSph=0))
    return model

r_check = np.geomspace(0.3, 300, 8)
model = halo_model(df_halo)
history = []
for iteration in range(10):
    model.iterate()                                   # density from the DF, then the new potential
    history.append(model.components[0].density.density(on_x_axis(r_check)))
history = np.array(history)
change = np.abs(history[1:] / history[:-1] - 1)       # relative change between consecutive iterations

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for k in range(len(r_check)):
    ax[0].semilogy(np.arange(1, 10), change[:, k], 'o-', ms=3, label=f'r = {r_check[k]:.1f} kpc')
ax[0].set_xlabel('iteration'); ax[0].set_ylabel('|relative change of $\\rho$|'); ax[0].legend(fontsize=7)
r_plot = np.geomspace(0.1, 500, 200)
ax[1].loglog(r_plot, agama.Density(type='Dehnen', gamma=1, mass=1e12, scaleRadius=30.).density(on_x_axis(r_plot)),
             'k:', label='initial guess (Hernquist)')
ax[1].loglog(r_plot, model.components[0].density.density(on_x_axis(r_plot)), 'C0-', label='self-consistent DPL halo')
ax[1].set_xlabel('$r$ [kpc]'); ax[1].set_ylabel('$\\rho$ [$M_\\odot$/kpc$^3$]'); ax[1].set_ylim(1e1, 1e10); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()
print('relative change in the last iteration, per radius:', np.round(change[-1], 5))
assert np.all(change[-1] < 1e-3)

**Figure 10.1.** Self-consistent double-power-law halo. Left: relative change of the density between consecutive iterations at eight radii; right: density profile of the initial Hernquist guess (dotted) and of the converged halo (solid).

Each iteration roughly halves the change (Figure 10.1, left), and after 10 iterations the density changes by less than $10^{-3}$
everywhere. The converged halo (Figure 10.1, right) has an inner slope close to $-1$ and an outer one close to $-3$, as designed.
(A practical note: the density is computed on a grid of radii. With a different grid — e.g. 30 points from 0.1 kpc —
one grid node in this model kept oscillating by ±5% instead of converging. If one radius refuses to converge, change
the grid.)

**Checks of self-consistency.** (1) The density obtained by integrating the DF over velocities in the final potential
(`GalaxyModel.moments`) should equal the density that generates that potential (`potential.density`, Poisson's equation).
(2) A sample of particles should satisfy the virial theorem $2K + W = 0$, where $K = \tfrac12\sum_i m_i |\mathbf v_i|^2$ is the
kinetic energy and $W = \tfrac12\sum_i m_i\Phi(\mathbf x_i)$ the potential energy of a self-gravitating system; the sums run
over the particles $i$, of mass $m_i$, position $\mathbf x_i$ and velocity $\mathbf v_i$, and $\Phi$ is the model potential.

In [ ]:
pot_halo = model.potential
gm_halo = agama.GalaxyModel(pot_halo, df_halo)
rho_df = gm_halo.moments(on_x_axis(r_check), dens=True, vel=False, vel2=False)
rho_poisson = pot_halo.density(on_x_axis(r_check))
print('DF density / Poisson density:', np.round(rho_df / rho_poisson, 4))
assert np.max(np.abs(rho_df / rho_poisson - 1)) < 0.01

xv, m = gm_halo.sample(100000)
K = 0.5 * np.sum(m * np.sum(xv[:, 3:]**2, axis=1))
W = 0.5 * np.sum(m * pot_halo.potential(xv[:, :3]))
print(f'virial ratio 2K/|W| = {2 * K / abs(W):.4f}')
assert abs(2 * K / abs(W) - 1) < 0.02

Both checks pass at the 1% level, which is the accuracy of the potential expansion and of the density
integrals.

## 10.3 A disk galaxy: quasi-isothermal disk + dark halo

**The quasi-isothermal DF** (Binney 2010; Binney & McMillan 2011) is built on the epicycle approximation of
Chapter 5. A near-circular orbit has radial energy $E_R \approx \kappa J_r$ and vertical energy $E_z \approx \nu J_z$, where $\kappa$ and $\nu$ are
the epicyclic and vertical frequencies (Chapter 5; here $\nu$ is a frequency, not a density), so a
Boltzmann-like distribution $e^{-E_R/\sigma_R^2}\, e^{-E_z/\sigma_z^2}$ becomes

$$ f(\mathbf J) = \frac{\Omega\,\Sigma}{2\pi^2\kappa^2}\,\frac{\kappa}{\sigma_R^2}\exp\!\left(-\frac{\kappa J_r}{\sigma_R^2}\right)
   \frac{\nu}{2\pi\sigma_z^2}\exp\!\left(-\frac{\nu J_z}{\sigma_z^2}\right) \;(\text{for } J_\phi > 0), $$

with $\Omega$ the circular frequency, $\Sigma$ the surface density of the disk, $\sigma_R$ and $\sigma_z$ its radial and vertical velocity
dispersions, and all of $\Omega, \kappa, \nu, \Sigma, \sigma_R, \sigma_z$ evaluated at the guiding radius $R_c(J_\phi)$, the radius of the
circular orbit with angular momentum $J_\phi$ (Chapter 5). The surface density
$\Sigma(R)$ and the dispersions are approximately exponential in radius. AGAMA's parameters: `mass` (or `Sigma0`),
`Rdisk`, `Hdisk` (scale height, which sets $\sigma_z$ through $\nu$), `sigmar0` and `Rsigmar` ($\sigma_R \approx$ `sigmar0` $e^{-R/{\rm Rsigmar}}$).
Since $\kappa$, $\nu$ depend on the potential, the DF needs a potential when it is created.

The model:

- **disk**: mass $M_d = 5\times10^{10}\,M_\odot$, scale length $R_d = 3$ kpc, scale height 0.3 kpc, $\sigma_R \approx 90\,e^{-R/6\,{\rm kpc}}$ km/s
  (≈ 24 km/s at the Sun);
- **halo**: the double power law of §10.2 with mass $6\times10^{11}\,M_\odot$.

As in AGAMA's example scripts, we start with fixed density profiles for both components, compute the potential,
create the disk DF in it, then replace both components by DF-based ones and iterate. A disk needs a 2-d grid in
$(R, z)$ (cylindrical radius and height above the plane) and an axisymmetric potential solver (`CylSpline`); the halo uses the spherical-harmonic one.

In [ ]:
M_disk, R_disk, h_disk = 5e10, 3.0, 0.3
M_halo = 6e11
galaxy = agama.SelfConsistentModel(rminSph=0.05, rmaxSph=1000, sizeRadialSph=40, lmaxAngularSph=8,
                                   RminCyl=0.1, RmaxCyl=40, sizeRadialCyl=30, zminCyl=0.03, zmaxCyl=10, sizeVerticalCyl=25,
                                   verbose=False)
disk_guess = agama.Density(type='Disk', surfaceDensity=M_disk / (2 * np.pi * R_disk**2), scaleRadius=R_disk,
                           scaleHeight=-h_disk)      # negative scale height = sech^2 vertical profile
halo_guess = agama.Density(type='Dehnen', gamma=1, mass=M_halo, scaleRadius=25.)
galaxy.components.append(agama.Component(density=disk_guess, disklike=True))
galaxy.components.append(agama.Component(density=halo_guess, disklike=False))
galaxy.iterate()                                      # potential of the initial guess

df_disk = agama.DistributionFunction(type='QuasiIsothermal', potential=galaxy.potential, mass=M_disk,
                                     Rdisk=R_disk, Hdisk=h_disk, sigmar0=90., Rsigmar=2 * R_disk, sigmamin=5.)
df_halo2 = agama.DistributionFunction(type='DoublePowerLaw', mass=M_halo, J0=5000., slopeIn=1.6, slopeOut=3.2, Jcutoff=1e5)
galaxy.components[0] = agama.Component(df=df_disk, disklike=True, RminCyl=0.2, RmaxCyl=30, sizeRadialCyl=20,
                                       zminCyl=0.05, zmaxCyl=5, sizeVerticalCyl=16)
galaxy.components[1] = agama.Component(df=df_halo2, disklike=False, rminSph=0.05, rmaxSph=500, sizeRadialSph=40,
                                       lmaxAngularSph=6)

R_check = np.array([2.0, 4.0, 8.0, 12.0, 20.0])
vc_history = []
for iteration in range(10):                          # each iteration takes a few seconds
    galaxy.iterate()
    vc_history.append(vcirc(galaxy.potential, R_check))
vc_history = np.array(vc_history)
for k in [0, 4, 9]:
    print(f'iteration {k + 1:2d}: v_c at R = {R_check} kpc: {np.round(vc_history[k], 1)} km/s')
dvc = np.max(np.abs(vc_history[-1] / vc_history[-2] - 1))
print(f'largest relative change of v_c in the last iteration: {dvc:.1e}')
assert dvc < 2e-3

The rotation curve settles after about ten iterations. Now look at the model (Figure 10.2): the rotation curve split into disk and
halo contributions (the potential of each component is computed from its density with a multipole or CylSpline
expansion), the surface density of the disk, its vertical profile at the solar radius, and its velocity dispersions.

In [ ]:
pot = galaxy.potential
dens_disk, dens_halo = galaxy.components[0].density, galaxy.components[1].density
pot_disk = agama.Potential(type='CylSpline', density=dens_disk, Rmin=0.1, Rmax=40, zmin=0.03, zmax=10, mmax=0)
pot_halo2 = agama.Potential(type='Multipole', density=dens_halo, lmax=6)
gm_disk = agama.GalaxyModel(pot, df_disk)

R = np.linspace(0.5, 25, 50)
fig, ax = plt.subplots(2, 2, figsize=(13, 9))
ax[0, 0].plot(R, vcirc(pot, R), 'k-', label='total')
ax[0, 0].plot(R, vcirc(pot_disk, R), 'C0--', label='disk')
ax[0, 0].plot(R, vcirc(pot_halo2, R), 'C1--', label='halo')
ax[0, 0].set_xlabel('$R$ [kpc]'); ax[0, 0].set_ylabel('$v_c$ [km/s]'); ax[0, 0].legend()

Sigma = gm_disk.moments(np.column_stack([R, 0 * R]), dens=True, vel=False, vel2=False)    # projected along z: surface density
ax[0, 1].semilogy(R, Sigma / 1e6, 'C0-', label='DF model')
ax[0, 1].semilogy(R, M_disk / (2 * np.pi * R_disk**2) * np.exp(-R / R_disk) / 1e6, 'k:', label='exponential, $R_d$ = 3 kpc')
ax[0, 1].set_xlabel('$R$ [kpc]'); ax[0, 1].set_ylabel('$\\Sigma$ [$M_\\odot$/pc$^2$]'); ax[0, 1].legend()

z = np.linspace(0, 2, 41)
rho_z = gm_disk.moments(np.column_stack([np.full_like(z, 8.0), 0 * z, z]), dens=True, vel=False, vel2=False)
ax[1, 0].semilogy(z, rho_z / 1e9, 'C0-', label='DF model at $R$ = 8 kpc')
ax[1, 0].semilogy(z, rho_z[0] / 1e9 / np.cosh(z / (2 * h_disk))**2, 'k:', label='sech$^2(z/2h)$, $h$ = 0.3 kpc')
ax[1, 0].set_xlabel('$z$ [kpc]'); ax[1, 0].set_ylabel('$\\rho$ [$M_\\odot$/pc$^3$]'); ax[1, 0].legend()

vmean, v2 = gm_disk.moments(on_x_axis(R), dens=False, vel=True, vel2=True)
ax[1, 1].plot(R, np.sqrt(v2[:, 0]), label='$\\sigma_R$')
ax[1, 1].plot(R, np.sqrt(v2[:, 2]), label='$\\sigma_z$')
ax[1, 1].plot(R, 90 * np.exp(-R / (2 * R_disk)), 'k:', label='90 exp$(-R/6)$')
ax[1, 1].set_xlabel('$R$ [kpc]'); ax[1, 1].set_ylabel('dispersion [km/s]'); ax[1, 1].legend()
plt.tight_layout(); plt.show()

i8 = np.argmin(np.abs(R - 8))
print(f'at R = 8 kpc: v_c = {vcirc(pot, 8.0)[0]:.1f} km/s (disk {vcirc(pot_disk, 8.0)[0]:.1f}, halo {vcirc(pot_halo2, 8.0)[0]:.1f}); '
      f'sigma_R = {np.sqrt(v2[i8, 0]):.1f}, sigma_z = {np.sqrt(v2[i8, 2]):.1f}, mean v_phi = {vmean[i8, 1]:.1f} km/s')
print(f'midplane disk density at 8 kpc: {rho_z[0] / 1e9:.3f} Msun/pc^3; surface density {Sigma[i8] / 1e6:.1f} Msun/pc^2')

# self-consistency: DF density of each component vs the density used for the potential
pts = np.array([[4, 0, 0], [8, 0, 0], [8, 0, 0.5], [12, 0, 0]])
ratio_disk = gm_disk.moments(pts, dens=True, vel=False, vel2=False) / dens_disk.density(pts)
print('disk: DF density / component density at (R, z) = (4,0), (8,0), (8,0.5), (12,0):', np.round(ratio_disk, 3))
assert np.max(np.abs(ratio_disk - 1)) < 0.02

**Figure 10.2.** The disk + halo galaxy after 10 iterations. Top left: rotation curve, total and the contributions of disk and halo; top right: disk surface density and an exponential with $R_d$ = 3 kpc; bottom left: vertical density profile at $R$ = 8 kpc and a sech$^2$ profile; bottom right: $\sigma_R$ and $\sigma_z$ of the disk and $90\,e^{-R/6\,{\rm kpc}}$ km/s.

A few things to notice in Figure 10.2:

- The rotation curve (top left) rises to about 210 km/s at 7 kpc and falls to about 170 km/s at 20 kpc. The disk dominates inside
  about 12 kpc and the halo outside. This is a rather disk-dominated galaxy; Exercise 3 asks you to make it more like
  the Milky Way of Chapter 4.
- The surface density (top right) and the dispersions (bottom right) are close to, but not exactly, exponential: the parameters of the
  quasi-isothermal DF are not the same as the properties of the resulting model, and need tuning when fitting data.
  Beyond 12 kpc the dispersions level off at the floor `sigmamin` = 5 km/s.
- The vertical profile (bottom left) follows the target sech$^2$ near the plane but falls faster above about 0.7 kpc. The sech$^2$ law
  is the profile of an isolated isothermal sheet; here the halo adds to the vertical force and $\sigma_z$ need not be
  constant with height, so the match is only approximate.
- The mean rotation is below the circular speed $v_c$: the asymmetric drift of Chapter 7, now produced automatically by the DF.

## 10.4 Adiabatic contraction of the halo

Compare the halo in the disk galaxy with the same halo DF on its own (§10.2's model, rescaled to the same mass). The
DF of the halo — the distribution of its particles over **actions** — is identical in both. But actions are adiabatic
invariants: the deeper potential of the disk pulls orbits with the same actions closer to the centre. The halo
**contracts** (Blumenthal et al. 1986 introduced the idea with a simpler invariant).

In [ ]:
halo_alone = halo_model(df_halo2)
for iteration in range(10):
    halo_alone.iterate()

r_plot = np.geomspace(0.3, 1000, 100)
rho_alone = halo_alone.components[0].density.density(on_x_axis(r_plot))
rho_with = dens_halo.density(on_x_axis(r_plot))
M_alone, M_with = halo_alone.components[0].density.enclosedMass(r_plot), np.array([dens_halo.enclosedMass(x) for x in r_plot])

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].loglog(r_plot, rho_alone, label='halo alone')
ax[0].loglog(r_plot, rho_with, label='halo with disk')
ax[0].set_xlabel('$r$ [kpc]'); ax[0].set_ylabel('$\\rho_{\\rm halo}$ [$M_\\odot$/kpc$^3$]'); ax[0].legend()
ax[1].semilogx(r_plot, M_with / M_alone)
ax[1].axhline(1, color='gray', ls=':')
ax[1].set_xlabel('$r$ [kpc]'); ax[1].set_ylabel('$M_{\\rm halo}(<r)$: with disk / alone')
plt.tight_layout(); plt.show()

for x in [2, 8, 30, 100, 1000]:
    print(f'halo mass inside {x:4d} kpc: with disk / alone = {np.interp(x, r_plot, M_with / M_alone):.3f}')
ratio = M_with / M_alone
assert ratio[np.searchsorted(r_plot, 2)] > 2 and np.all(np.diff(ratio[r_plot > 1]) < 0) and ratio[-1] < 1.05

**Figure 10.3.** The same halo DF alone and inside the disk galaxy. Left: halo density profiles; right: halo mass inside $r$ with the disk divided by that without it.

The halo mass inside 2 kpc grows by a factor of about 3 (Figure 10.3, right; the largest factor, 3.5, is near 1 kpc), inside 8 kpc by about 2.
The effect decreases outwards but is
still 16% at 100 kpc. This halo is very extended (only a third of its mass lies inside 100 kpc), and the disk deepens
the potential everywhere, so even distant orbits become slightly more bound; the ratio is still 1.03 at
1000 kpc. Contraction matters when dark-matter halos from simulations without baryons are compared with real
galaxies.

## Summary

- DFs of actions are automatically steady, describe any axisymmetric component with one formula, and stay fixed
  when the potential changes slowly.
- Double-power-law DFs describe halos and bulges; quasi-isothermal DFs describe disks, built on epicyclic frequencies.
- A self-consistent model is found by iterating DF → density → potential; check convergence and compare the DF density
  with the Poisson density.
- With fixed $f(\mathbf J)$, a halo contracts when a disk is added.

| task | AGAMA |
|---|---|
| halo / bulge DF | `agama.DistributionFunction(type='DoublePowerLaw', mass=, J0=, slopeIn=, slopeOut=, ...)` |
| disk DF | `agama.DistributionFunction(type='QuasiIsothermal', potential=, mass=, Rdisk=, Hdisk=, sigmar0=, Rsigmar=)` |
| self-consistent model | `model = agama.SelfConsistentModel(...)`; `model.components.append(agama.Component(df=, disklike=, ...))`; `model.iterate()` |
| result | `model.potential`, `model.components[i].density`; then `agama.GalaxyModel(model.potential, df)` for moments and samples |

## Exercises

1. **Anisotropic halo.** Set `coefJrOut=1.5` (and `coefJzOut=0.75`) in the halo DF of §10.2. Compute the anisotropy $\beta(r)$ (Chapter 7) from the moments.
2. **Hotter disk.** Double `sigmar0`. How do $\Sigma(R)$, the asymmetric drift at 8 kpc and the halo contraction change?
3. **A Milky Way.** Tune `M_halo` and the disk parameters so that $v_c(8\,{\rm kpc}) \approx 230$ km/s and the rotation curve agrees with
   the Eilers et al. data of Chapter 4 within 5% between 5 and 20 kpc.
4. **Sample and evolve.** Sample 20 000 disk stars from `gm_disk`, integrate them for 1 Gyr in `pot`, and check that the vertical
   density profile at $7 < R < 9$ kpc does not change (Chapter 8 §8.3).

## References

- J. Binney (2010), "Distribution functions for the Milky Way", MNRAS 401, 2318: https://ui.adsabs.harvard.edu/abs/2010MNRAS.401.2318B
- J. Binney & P. McMillan (2011), "Models of our Galaxy - II", MNRAS 413, 1889: https://ui.adsabs.harvard.edu/abs/2011MNRAS.413.1889B
- L. Posti et al. (2015), "Action-based distribution functions for spheroidal galaxies", MNRAS 447, 3060:
  https://ui.adsabs.harvard.edu/abs/2015MNRAS.447.3060P
- E. Vasiliev (2019), AGAMA, MNRAS 482, 1525: https://doi.org/10.1093/mnras/sty2672
- G. R. Blumenthal et al. (1986), "Contraction of dark matter galactic halos due to baryonic infall", ApJ 301, 27:
  https://ui.adsabs.harvard.edu/abs/1986ApJ...301...27B
- J. Binney & S. Tremaine, *Galactic Dynamics*, 2nd ed. (Princeton, 2008), §4.6.